In [1]:
# Cell 1: Imports and Path Setup
import sys
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict, field
import pickle

# Add parent directory for imports
# sys.path.insert(0, str(Path.cwd().parent))
# os.chdir(Path.cwd().parent)

# Vast.ai cloud serversetup
sys.path.insert(0, str(Path.cwd() / "workspace" / "trade-automation"))
os.chdir(Path.cwd() / "workspace" / "trade-automation")

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# !git pull

In [3]:


# # LSTM modules
from crypto_analysis.lstm.model import ModelConfig, CNNLSTMSignalPredictor
from crypto_analysis.lstm.trainer import Trainer, TrainingConfig, TrainingHistory
from crypto_analysis.lstm.loss import BinarySignalLoss, FocalBinaryLoss
from crypto_analysis.lstm.dataset import SignalDataset, create_sequences

# VectorBT Data Preprocessor (replaces DataPreprocessor and DatasetBuilder)
from crypto_analysis.vectorbt_optimizer import VectorBTDataPreprocessor


# MLflow
import mlflow
# import mlflow.pytorch

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MLflow version: {mlflow.__version__}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5070
MLflow version: 3.9.0


In [4]:
# Cell 2: Configuration Constants

# === CONFIGURATION ===

# Data paths - VectorBT optimized CSV files
CSV_DIR = Path("notebooks/csvs")  # Directory with vectorbt_optimizer output CSVs
OUTPUT_DIR = Path("notebooks/coin_csvs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

# VectorBTDataPreprocessor configuration
PREPROCESSOR_CONFIG = {
    "remove_raw_indicators": True,   # Keep only OHLCV + *_entry/*_exit signals
    "target_shift": 1,               # Features at t predict target at t+1
    "sequence_length": 18,           # LSTM input sequence length
    "stride": 1,                     # Step between sequences
    "scaler_type": "minmax",       # 'standard' or 'minmax'
    "target_column": "tradeable",
    "normalize_by_close": True,
}

# Train/val/test split ratios
SPLIT_CONFIG = {
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "test_ratio": 0.2,
}

# MLflow experiment name
MLFLOW_EXPERIMENT = "multi_coin_lstm_training"

print("Configuration loaded.")
print(f"CSV directory: {CSV_DIR.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Sequence length: {PREPROCESSOR_CONFIG['sequence_length']}")
print(f"Target shift: {PREPROCESSOR_CONFIG['target_shift']}")
print(f"Remove raw indicators: {PREPROCESSOR_CONFIG['remove_raw_indicators']}")

Configuration loaded.
CSV directory: /workspace/trade-automation/notebooks/csvs
Output directory: /workspace/trade-automation/notebooks/coin_csvs
Sequence length: 18
Target shift: 1
Remove raw indicators: True


In [5]:
from crypto_analysis.vectorbt_optimizer import optimize_all
results = optimize_all(
    symbols="whitelist",
    indicators="all",
    data_dir="data/binance",
    output_dir="notebooks/test_csv",
    config_path="config.json",
    n_processes=10,
    n_jobs_optuna=4,
    threshold_pct=2,
    period_hours=6,
    export_csv=True,
    export_params_json=True
)

Data not found for MATIC: Data file not found: data/binance/MATIC_USDT-1h.feather
/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/parallel_runner.py:111: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/parallel_runner.py:111: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/parallel_runner.py:111: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.

Username for 'https://github.com': ^C


In [103]:
preprocessor = VectorBTDataPreprocessor(
    extract_time_features= True,
    enable_dataframe_clustering = True,
    df_cluster_columns="indicators",
    enable_signal_clustering = True,
    keep_original_signals = True,
    enable_derived_features=True,
    enable_derived_clustering=True,
    enable_resampling=True,
    train_ratio=0.8,
    val_ratio=0.1,
    remove_raw_indicators=PREPROCESSOR_CONFIG["remove_raw_indicators"],
    target_shift=PREPROCESSOR_CONFIG["target_shift"],
    sequence_length=PREPROCESSOR_CONFIG["sequence_length"],
    stride=PREPROCESSOR_CONFIG["stride"],
    scaler_type=PREPROCESSOR_CONFIG["scaler_type"],
    target_column=PREPROCESSOR_CONFIG["target_column"],
    normalize_by_close=PREPROCESSOR_CONFIG["normalize_by_close"],
)

results = preprocessor.load_csv("notebooks/test_csv/AVAX_optimized.csv")
# Get transformed DataFrame with all cluster columns
partition_pct = 0.5
slice_ind = int(results.shape[0] - results.shape[0]*partition_pct)
results = results.iloc[slice_ind:,:]
print("Original results shape:", results.shape)
transformed_df = preprocessor.fit_transform_dataframe(results, apply_resampling=True, apply_scaling=True)
cluster_columns = [c for c in transformed_df.columns if 'cluster' in c]
entr_exit_columns = [c for c in transformed_df.columns if c.endswith("_entry") or c.endswith("_exit")]
derived_columns = [c for c in transformed_df.columns if c.startswith("derived_") and c.startswith("derived_cluster")==False]
time_columns = ["day_sin", "day_cos", "hour_sin", "hour_cos"]
ohlvc = ["tradeable", "open", "high", "low", "close", "volume", "split"]

# print("cluster columns:", cluster_columns)
# print("transformed_df columns:", list(transformed_df.columns))  # Debug: see what's available

# # Now use transformed_df for your feature selection
# feats = json.loads(open("notebooks/doge_feats.json").read())
# feats = [c for c in feats if c in transformed_df.columns]  # <-- CHANGE THIS LINE
# feats = list(set(feats + cluster_columns + ["tradeable", "day_sin", "day_cos", "hour_sin", "hour_cos"]))
# # Also filter final list against transformed_df columns
# feats = [c for c in feats if c in transformed_df.columns]
print(transformed_df.shape)
df = transformed_df[cluster_columns + entr_exit_columns + time_columns + ohlvc + derived_columns]
df.dropna(inplace=True)
n_features = len(df.columns)
print(df.shape)
print("Number of features", n_features)


Original results shape: (4601, 198)


/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/data_preprocessor.py:1196: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/data_preprocessor.py:1197: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/workspace/trade-automation/crypto_analysis/vectorbt_optimizer/data_preprocessor.py:1202: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

(4600, 326)
(4599, 254)
Number of features 254


/tmp/ipykernel_3098/606603740.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [104]:
df.groupby("split")["tradeable"].value_counts(normalize=True)

split  tradeable
test   hold         0.746204
       trade        0.253796
train  hold         0.769984
       trade        0.230016
val    hold         0.793478
       trade        0.206522
Name: proportion, dtype: float64

In [ ]:
# df_train = df[df["split"] == "train"] 
# df_val = df[df["split"] == "val"]
# df_test = df[df["split"] == "test"]
# df_train = pd.concat([df_train, df_val])
# df_val = df_test.copy()
# df_val["split"] = "val"
# df_train["split"] = "train"
# df = pd.concat([df_train, df_val, df_test])

In [105]:
train_dataset, val_dataset, test_dataset = preprocessor.create_sequences_by_split(df, 84, device="cuda")

In [112]:
# Cell 9: Create Model Configuration

model_config = ModelConfig(
    input_size=n_features-2,
    hidden_size=160, # 162
    num_layers=1,
    dropout=0.3058918689728847388,
    bidirectional=False,
    num_classes=2,
    input_seq_length=PREPROCESSOR_CONFIG["sequence_length"],
    classifier_hidden_size=40,
    # CNN-LSTM specific
    kernel_size=7,
    cnn_num_layers=3,
    cnn_dropout=0.30528918689728847388,
    lstm_dropout=0.305228918689728847388,
    classifier_dropout=0.305228918689728847388,
)

print("Model Configuration:")
print("=" * 40)
for field_name, value in asdict(model_config).items():
    print(f"  {field_name}: {value}")
# Cell 10: Create Model

model = CNNLSTMSignalPredictor(model_config)
print(model)
print(f"\nTotal trainable parameters: {model.get_num_parameters():,}")

Model Configuration:
  input_size: 252
  hidden_size: 160
  num_layers: 1
  dropout: 0.30589186897288473
  bidirectional: False
  num_classes: 2
  input_seq_length: 18
  classifier_hidden_size: 40
  kernel_size: 7
  cnn_num_layers: 3
  cnn_dropout: 0.30528918689728846
  lstm_dropout: 0.30522891868972885
  classifier_dropout: 0.30522891868972885
CNNLSTMSignalPredictor(
  input_size=252,
  projection_size=504,
  hidden_size=160,
  cnn_num_layers=3,
  kernel_size=7,
  cnn_dropout=0.30528918689728846,
  lstm_num_layers=1,
  lstm_dropout=0.30522891868972885,
  classifier_hidden_size=40,
  classifier_dropout=0.30522891868972885,
  bidirectional=False,
  num_classes=2 (hold=0, trade=1),
  total_params=960,034
)

Total trainable parameters: 960,034


In [ ]:
# Cell 12: Create Training Configuration

training_config = TrainingConfig(
    
    # Training parameters
    epochs=150,
    batch_size=8,
    learning_rate=0.0010597632290035332,
    weight_decay=0.0001,
    optimizer='adamw',
    grad_clip_norm=1.0,
    
    # Learning rate scheduler
    scheduler='plateau',
    scheduler_patience=10,
    scheduler_factor=0.85,

    # Data expansion
    progressive_expansion = True,
    expansion_accuracy_threshold = 0.95,
    expansion_consecutive_epochs = 2,
    expansion_partition_count = 6,
    expansion_max_partitions = 2,
    expansion_shuffle_on_add=False,
    
    # Class imbalance handling (from helper function)
    auto_class_weights=True,
    class_weight_power=0.506503212404369325,
    focal_loss=False,
    focal_gamma=1.518868882328869,
    label_smoothing=0.071028878700085323628,
    
    # Data split (we provide pre-split datasets)
    val_split=0.2,
    test_split=0.2,

    shuffle_every_epoch=True,
    
    # Early stopping
    early_stopping=False,
    patience=40,
    min_delta=1e-4,
    
    # Checkpointing
    checkpoint_dir=str(OUTPUT_DIR / "checkpoints"),
    save_best_only=True,
    save_interval=300,
    
    # Device
    device='cuda',
    
    # Logging
    log_interval=50,
    verbose=True,
)

print("Training Configuration created.")
print(f"Epochs: {training_config.epochs}")
print(f"Batch size: {training_config.batch_size}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Focal loss: {training_config.focal_loss}")
print(f"Class weight power: {training_config.class_weight_power}")

Training Configuration created.
Epochs: 200
Batch size: 8
Learning rate: 0.0010597632290035332
Focal loss: False
Class weight power: 0.5065032124043694


In [108]:
# Cell 13: MLflow Setup and Logging Functions

def setup_mlflow(experiment_name: str) -> str:
    """
    Setup MLflow experiment.
    
    Parameters
    ----------
    experiment_name : str
        Name for the MLflow experiment
    
    Returns
    -------
    str
        Experiment ID
    """
    # Set tracking URI (default is local ./mlruns)
    mlflow.set_tracking_uri("http://localhost:5000")
    
    # Create or get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
    else:
        experiment_id = experiment.experiment_id
    
    mlflow.set_experiment(experiment_name)
    
    print(f"MLflow experiment: {experiment_name}")
    print(f"Experiment ID: {experiment_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    
    return experiment_id


def log_training_params(
    model_config: ModelConfig,
    training_config: TrainingConfig,
    preprocessor_config: Dict,
    symbols: List[str],
):
    """
    Log all configuration parameters to MLflow.
    """
    # Model params
    mlflow.log_params({
        "model.input_size": model_config.input_size,
        "model.hidden_size": model_config.hidden_size,
        "model.num_layers": model_config.num_layers,
        "model.dropout": model_config.dropout,
        "model.bidirectional": model_config.bidirectional,
        "model.classifier_hidden_size": model_config.classifier_hidden_size,
        "model.kernel_size": model_config.kernel_size,
        "model.cnn_num_layers": model_config.cnn_num_layers,
        "model.cnn_dropout": model_config.cnn_dropout,
        "model.lstm_dropout": model_config.lstm_dropout,
    })
    
    # Training params
    mlflow.log_params({
        "train.epochs": training_config.epochs,
        "train.batch_size": training_config.batch_size,
        "train.learning_rate": training_config.learning_rate,
        "train.weight_decay": training_config.weight_decay,
        "train.optimizer": training_config.optimizer,
        "train.scheduler": training_config.scheduler,
        "train.focal_loss": training_config.focal_loss,
        "train.focal_gamma": training_config.focal_gamma,
        "train.class_weight_power": training_config.class_weight_power,
        "train.label_smoothing": training_config.label_smoothing,
        "train.early_stopping": training_config.early_stopping,
        "train.patience": training_config.patience,
    })
    
    # Preprocessor params
    mlflow.log_params({
        "data.sequence_length": preprocessor_config["sequence_length"],
        "data.target_shift": preprocessor_config["target_shift"],
        "data.stride": preprocessor_config["stride"],
        "data.remove_raw_indicators": preprocessor_config["remove_raw_indicators"],
        "data.num_coins": len(symbols),
    })
    


def log_training_metrics(history: TrainingHistory, epoch: int):
    """
    Log training metrics for a single epoch.
    """
    mlflow.log_metrics({
        "train_loss": history.train_losses[-1],
        "val_loss": history.val_losses[-1],
        "train_accuracy": history.train_accuracies[-1],
        "val_accuracy": history.val_accuracies[-1],
        "learning_rate": history.learning_rates[-1],
    }, step=epoch)


def log_evaluation_results(results: Dict[str, Dict]):
    """
    Log final evaluation metrics.
    """
    for split_name, metrics in results.items():
        if metrics is None:
            continue
        prefix = f"{split_name}_"
        mlflow.log_metrics({
            f"{prefix}accuracy": metrics['accuracy'],
            f"{prefix}precision": metrics['precision'],
            f"{prefix}recall": metrics['recall'],
            f"{prefix}f1": metrics['f1'],
            f"{prefix}hold_f1": metrics['hold_f1'],
            f"{prefix}trade_f1": metrics['trade_f1'],
        })


def log_model_artifact(model: torch.nn.Module, preprocessor: VectorBTDataPreprocessor, output_dir: Path):
    """
    Log model and preprocessor as MLflow artifacts.
    """
    # Log PyTorch model
    mlflow.pytorch.log_model(model, "model")
    
    # Log preprocessor
    preprocessor_path = output_dir / "preprocessor_mlflow.pkl"
    preprocessor.save(preprocessor_path)
    mlflow.log_artifact(str(preprocessor_path))
    preprocessor_path.unlink()  # Clean up temp file

In [109]:
experiment_id = setup_mlflow(MLFLOW_EXPERIMENT)

MLflow experiment: multi_coin_lstm_training
Experiment ID: 1
Tracking URI: http://localhost:5000


In [110]:
# Cell 15: Training Function with MLflow

def train_with_mlflow(
    model: torch.nn.Module,
    training_config: TrainingConfig,
    train_dataset: SignalDataset,
    val_dataset: SignalDataset,
    test_dataset: SignalDataset,
    model_config: ModelConfig,
    preprocessor_config: Dict,
    symbols: List[str],
    preprocessor: VectorBTDataPreprocessor,
    output_dir: Path
) -> Tuple[TrainingHistory, Dict, Trainer, str]:
    """
    Train model with full MLflow tracking.
    
    Returns
    -------
    Tuple[TrainingHistory, Dict, Trainer, str]
        (history, evaluation_results, trainer, run_id)
    """
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        print("=" * 60)
        
        # Log all parameters
        log_training_params(
            model_config, training_config, 
            preprocessor_config,
            symbols
        )
        
        # Create trainer (use preprocessor for consistency, though VectorBTDataPreprocessor
        # doesn't have the same interface as DataPreprocessor - we'll pass None)
        trainer = Trainer(
            model=model,
            config=training_config,
            preprocessor=None  # VectorBTDataPreprocessor has different interface
        )
        
        # Store test dataset for evaluation
        trainer.test_dataset = test_dataset
        
        # Define callback for epoch-level logging
        def mlflow_callback(epoch: int, history: TrainingHistory):
            log_training_metrics(history, epoch)
        
        # Train
        print("\nStarting training...")
        history = trainer.train(
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            callbacks=[mlflow_callback]
        )
        
        # Evaluate
        print("\nEvaluating model...")
        results = trainer.evaluate_all(verbose=True)
        
        # Log evaluation metrics
        log_evaluation_results(results)
        
        # Log final metrics
        mlflow.log_metrics({
            "best_epoch": history.best_epoch + 1,
            "best_val_loss": history.best_val_loss,
            "epochs_trained": len(history.train_losses),
        })
        
        # Log model artifact
        log_model_artifact(model, preprocessor, output_dir)
        
        print(f"\nMLflow Run completed: {run_id}")
        
    return history, results, trainer, run_id

In [111]:

# Get symbols list
symbols = list(results.keys())

# Execute training with MLflow
history, eval_results, trainer, run_id = train_with_mlflow(
    model=model,
    training_config=training_config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    model_config=model_config,
    preprocessor_config=PREPROCESSOR_CONFIG,
    symbols=symbols,
    preprocessor=preprocessor,
    output_dir=OUTPUT_DIR
)

print(f"\nTraining complete!")
print(f"Best epoch: {history.best_epoch + 1}")
print(f"Best validation loss: {history.best_val_loss:.4f}")
print(f"MLflow Run ID: {run_id}")

MLflow Run ID: 3fc76ef6625e43a18497c7d6afa28bcc

Starting training...
Progressive expansion: divided train data into 6 partitions
  Partition 1: 599 samples
  Partition 2: 599 samples
  Partition 3: 599 samples
  Partition 4: 599 samples
  Partition 5: 599 samples
  Partition 6: 598 samples
Starting with partition 1 (599 samples)
Class weights: hold=0.703, trade=1.297
Training on cuda
Train samples: 599, Val samples: 375, Test samples: 376
Batches per epoch: 75
--------------------------------------------------
  Batch 0/75, Loss: 0.9659
  Batch 50/75, Loss: 0.7785
Epoch 1/200
  Train Loss: 0.7451, Train Acc: 0.5793
  Val Loss: 0.6484, Val Acc: 0.7787
  LR: 1.06e-03
  Saved best model (val_loss: 0.6484)
  Batch 0/75, Loss: 0.8755
  Batch 50/75, Loss: 0.6662
Epoch 2/200
  Train Loss: 0.7153, Train Acc: 0.6144
  Val Loss: 0.6731, Val Acc: 0.7787
  LR: 1.06e-03
  Batch 0/75, Loss: 0.7344
  Batch 50/75, Loss: 0.9255
Epoch 3/200
  Train Loss: 0.6803, Train Acc: 0.6678
  Val Loss: 0.6464, Va

2026/01/30 20:15:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



COMPREHENSIVE MODEL EVALUATION REPORT (Binary: hold=0, trade=1)

--------------------------------------------------------------------------------
OVERALL METRICS COMPARISON
--------------------------------------------------------------------------------
Metric          TRAIN                VAL                  
--------------------------------------------------------------------------------
Accuracy        1.0000               0.9379               
Precision       1.0000               0.9382               
Recall          1.0000               0.9379               
F1              1.0000               0.9380               

--------------------------------------------------------------------------------
TRAIN DATASET - Per-Class Metrics
--------------------------------------------------------------------------------
Class      Precision    Recall       F1           Support   
------------------------------------------------------------
hold       1.0000       1.0000       1.0000       

2026/01/30 20:15:23 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/01/30 20:15:23 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/30 20:15:29 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to mak


MLflow Run completed: 3fc76ef6625e43a18497c7d6afa28bcc
🏃 View run thoughtful-trout-879 at: http://localhost:5000/#/experiments/1/runs/3fc76ef6625e43a18497c7d6afa28bcc
🧪 View experiment at: http://localhost:5000/#/experiments/1

Training complete!
Best epoch: 132
Best validation loss: 0.3007
MLflow Run ID: 3fc76ef6625e43a18497c7d6afa28bcc


In [ ]:
# from crypto_analysis.lstm_optimizer import LSTMMetaheuristicOptimizer, HyperparamConfig
# LSTMMetaheuristicOptimizer.CNN_LSTM_HYPERPARAM_CONFIGS = [
#         # === Model architecture params (ModelConfig) ===
#         HyperparamConfig('hidden_size', 64, 256, 'int', 'hidden_size'),
#         HyperparamConfig('num_layers', 1, 3, 'int', 'num_layers'),
#         HyperparamConfig('dropout', 0.15, 0.30, 'float', 'dropout'),  # Used for cnn/lstm/classifier dropout
#         HyperparamConfig('classifier_hidden_size', 16, 64, 'int', 'classifier_hidden_size'),
#         HyperparamConfig('input_seq_length', 12, 24, 'int', 'input_seq_length'),
#         # CNN-specific params (kernel_size maps to odd values: 1->3, 2->5, 3->7, etc.)
#         HyperparamConfig('kernel_size', 1, 3, 'int', 'kernel_size'),
#         HyperparamConfig('cnn_num_layers', 1, 3, 'int', 'cnn_num_layers'),
#         # === Training params (TrainingConfig) ===
#         HyperparamConfig('learning_rate', 0.001, 0.01, 'float', 'learning_rate'),
#         HyperparamConfig('weight_decay', 0.0001, 0.001, 'float', 'weight_decay'),
#         HyperparamConfig('batch_size', 512, 512, 'int', 'batch_size'),
#         HyperparamConfig('scheduler_patience', 15, 25, 'int', 'scheduler_patience'),
#         # === Class imbalance handling (TrainingConfig) ===
#         HyperparamConfig('class_weight_power', 0.2, 0.7, 'float', 'class_weight_power'),
#         HyperparamConfig('focal_gamma', 1.0, 3.0, 'float', 'focal_gamma'),
#         HyperparamConfig('label_smoothing', 0.01, 0.15, 'float', 'label_smoothing'),
#     ]

In [ ]:
# optimizer = LSTMMetaheuristicOptimizer(
#     df=results,  # Use DOGE data for optimization
#     preprocessor_type='vectorbt',  # Use new VectorBT preprocessor
#     enable_mlflow=False,
#     mlflow_experiment_name='lstm_crypto_optimization',
#     mlflow_tracking_uri='http://127.0.0.1:5000',  # Optional
#     pop_size=8,
#     iterations=70,
#     n_workers=10,
#     np_neighbors=1,
#     pf_max=0.25,
#     elitist_selection=False,
#     model_type="cnn_lstm",
#     epochs_per_eval=125,
#     normalize_by_close=True,
# )
# result = optimizer.optimize()
